# Configuration

Immune works with no configuration at all. When you want to pin a site's profile, enforce a detector or share state
between workers, you configure it in one of three places:

| Where | How | Best for |
| --- | --- | --- |
| An `immune.yaml` file | `immune.init()` finds `./immune.yaml`, or `immune.init(config="path/to/immune.yaml")` | Production: reviewed and versioned with your code |
| A Python dict | `immune.init(config={...})` | Tests, notebooks, settings computed at startup |
| Environment variables | `IMMUNE_MODE`, `IMMUNE_CONFIG`, `IMMUNE_STATE_URL`, `IMMUNE_DISABLED`, `IMMUNE_HOME` | Per-deployment overrides |

**Precedence**, lowest to highest: built-in defaults → the file or dict → `IMMUNE_MODE` and `IMMUNE_STATE_URL` →
keyword arguments to `immune.init()` (`mode=`, `state_dir=`).

## 1. A complete `immune.yaml` for a real app

This is the configuration for a store with a customer chat that uses tools, a help center, and a background
classifier. Every key is optional.

In [1]:
import tempfile
from pathlib import Path

import immune

project = Path(tempfile.mkdtemp(prefix="pinecrest-"))  # stands in for your project folder
(project / "immune.yaml").write_text("""
mode: auto

sites:
  support-chat:                        # customer chat with tools
    archetype: customer_service
    user_facing: true
    organs: [business]
    enforce: [input.off_task]
    allowed_destinations: [pinecrest.example]
    tools:
      issue_refund: {writes_state: true, irreversible: true}
      send_receipt: {egress: true}
  help-center:                         # answers from the knowledge base
    user_facing: true
    enforce: [input.off_task]
  review-triage:                       # structured extraction in a batch job
    user_facing: false
    echo: {enforce: true}

limits:
  max_identical_tool_calls: 2

privacy:
  verdict_log: logs/verdicts.jsonl     # relative paths resolve against this file
""")

runtime = immune.init(config=project / "immune.yaml")
settings = runtime.settings
print("mode:          ", settings.mode)
print("support-chat:  ", settings.sites["support-chat"].enforce, settings.sites["support-chat"].tools["issue_refund"])
print("tool limit:    ", settings.limits.max_identical_tool_calls)
print("verdict log:   ", settings.privacy.verdict_log.relative_to(project))

mode:           auto
support-chat:   ('input.off_task',) egress=None writes_state=True executes=None reads_private=None irreversible=True allowed_destinations=()
tool limit:     2
verdict log:    logs/verdicts.jsonl


Where it goes in your project:

```
my-app/
├── immune.yaml          ← found automatically when the app starts in this folder
├── vaccines/            ← your own protections (see the Vaccines guide)
├── app/
│   └── main.py          ← immune.init() at startup
└── tests/
```

Call `immune.init()` once, where your app starts:

| Framework | Where |
| --- | --- |
| A script or CLI | At the top of `main()` |
| FastAPI | In the `lifespan` handler, before `yield` |
| Django | In your `AppConfig.ready()` |
| Flask | In `create_app()` |
| Celery | In a `worker_process_init` signal handler (once per worker process) |
| Gunicorn with `preload_app` | In `post_fork`, so each worker has its own runtime |

## 2. Validate a file before you deploy

`immune config validate` checks a file against the schema and prints the settings it resolves. Add it to CI.
(In a terminal, drop the `!`.)

In [2]:
import tempfile
from pathlib import Path

path = Path(tempfile.mkdtemp()) / "immune.yaml"
path.write_text("mode: observe\nsites:\n  support-chat:\n    enforce: [input.off_task]\n")

!immune config validate {path}

valid configuration (mode=observe, 1 sites)


## 3. What a mistake looks like

Unknown keys and wrong values fail at startup with `immune.ConfigError`, naming the field, so a typo can't silently
switch a protection off:

In [3]:
import immune

try:
    immune.init(config={"mode": "auto", "sites": {"support-chat": {"enforced": ["input.off_task"]}}})
except immune.ConfigError as error:
    print(str(error)[:600])

invalid immune configuration: 1 validation error for Settings
sites.support-chat.enforced
  Extra inputs are not permitted [type=extra_forbidden, input_value=['input.off_task'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden


In [4]:
import immune

try:
    immune.init(config={"mode": "paranoid"})
except immune.ConfigError as error:
    print(str(error)[:400])

invalid immune configuration: 1 validation error for Settings
mode
  Input should be 'auto', 'observe', 'strict' or 'off' [type=enum, input_value='paranoid', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/enum


## 4. Every setting and its default

Printed from the running version, so this list is always current. Omit anything you don't need to change.

In [5]:
import yaml

import immune

runtime = immune.init(config={})
defaults = runtime.settings.model_dump(mode="json", exclude={"state_dir"})
print(yaml.safe_dump(defaults, sort_keys=False, width=120))

mode: auto
on_internal_error: pass
on_block: respond
streaming: progressive
canary: true
sensor:
  model: jev-1.13.0
  timeout_ms: 800
  api_key: null
  api_keys: []
  requests_per_minute: 1200
  coalesce: false
  breaker_error_rate: 0.2
  breaker_window_s: 60.0
  breaker_cooldown_s: 30.0
  deadline_margin_ms: 250
  on_outage: act_on_candidates
privacy:
  redact_before_sensor: true
  log: redacted
  profiling: jev
  verdict_log: null
promotion:
  enabled: true
  max_firing_rate: 0.001
  min_calls: 5000
  min_days: 14.0
limits:
  max_identical_tool_calls: 3
  max_tool_calls_per_turn: 25
  session_ttl_s: 86400.0
  max_sessions: 50000
state:
  backend: local
  url: null
  path: null
  key_prefix: 'immune:'
  failover_cooldown_s: 30.0
telemetry:
  opentelemetry: true
  alerts:
    enabled: true
    window_s: 900.0
    ratio: 5.0
    min_fired: 5
    absolute_rate: 0.05
    cooldown_s: 3600.0
  langsmith:
    enabled: auto
    project: null
    inputs: masked
    jev_runs: true
    feedback

The groups:

| Group | What it controls |
| --- | --- |
| `mode`, `on_block`, `on_internal_error`, `streaming`, `canary` | Global behavior |
| `sensor` | Jev: model, timeouts, keys, rate limits, circuit breaker, `on_outage` |
| `privacy` | What is masked before Jev, what logs keep, the optional verdict log |
| `promotion` | When observed detectors are promoted per site |
| `limits` | Tool-call loops and session memory |
| `state` | Where sessions, provenance and promotions are stored (`local`, `sqlite`, `redis`) |
| `telemetry` | OpenTelemetry, alerts and LangSmith |
| `heads` | A trained heads artifact from `immune evidence fit` |
| `vaccines` | Where your protections live, and which built-in ones are switched off |
| `sites` | Everything per site: profile, posture, tools, destinations |
| `endpoints` | Extra routes for gateways that don't use provider paths |

The [configuration reference](../../docs/guides/configuration.md) explains every key, and
`immune config schema` prints the JSON Schema.

## 5. Environment variables

| Variable | Effect |
| --- | --- |
| `TYPESAFE_API_KEY` | The Jev key. Keep keys out of YAML |
| `IMMUNE_MODE` | Overrides `mode` for this deployment, such as `observe` in staging |
| `IMMUNE_CONFIG` | Path of the configuration file, when it isn't `./immune.yaml` |
| `IMMUNE_STATE_URL` | Shared state: `redis://…`, `rediss://…` or `sqlite:///path` |
| `IMMUNE_DISABLED` | `1` makes `immune.init()` do nothing: the kill switch |
| `IMMUNE_HOME` | Where local state is kept (default `~/.immune`) |

In [6]:
import os

import immune

os.environ["IMMUNE_MODE"] = "observe"
print("with IMMUNE_MODE=observe:", immune.init(config={"mode": "strict"}).settings.mode)
del os.environ["IMMUNE_MODE"]
print("without it:              ", immune.init(config={"mode": "strict"}).settings.mode)

with IMMUNE_MODE=observe: observe
without it:               strict


## 6. Gateways and custom endpoints

Immune recognizes the OpenAI, Azure OpenAI, Anthropic, Gemini, Vertex and Bedrock APIs, and OpenAI-compatible servers
on standard paths (`/v1/chat/completions`). If your calls go through a gateway on a custom path, add a route so
Immune knows which wire format it speaks:

```yaml
endpoints:
  - openai_chat=^/llm/complete$         # <format>=<path regex>
  - anthropic_messages=^/proxy/claude$
```

Formats: `openai_chat`, `openai_responses`, `anthropic_messages`, `gemini`. `immune.coverage()` lists what is
intercepted in your process.

In [7]:
import immune

immune.init()
coverage = immune.coverage()
print("intercepted at:", coverage.points)
print("adapters:      ", coverage.adapters)
print("warnings:      ", coverage.warnings or "none")

intercepted at: ['httpx2: every client transport', 'httpx: every client transport']
adapters:       ['google-genai async calls routed through httpx', 'DSPy built-in transport', 'boto3 bedrock-runtime Converse and ConverseStream']
warnings:       none


## Recap

- `immune.yaml` next to your app, or `immune.init(config=...)`; environment variables override per deployment
- Mistakes fail fast with `immune.ConfigError`; `immune config validate` checks files in CI
- Every default is printed above from the running version
- `endpoints` routes gateways on custom paths

Next: [Untrusted data and RAG](09_untrusted_data_and_rag.ipynb)